# Stage 4 · Reward Design & Shaping — EXERCISES
### Topics: Rule-Based vs Learned Rewards · Reward Hacking · PRMs · ORMs · Format Rewards · Verifiable Rewards

> Fill every `# TODO`. Run `# ASSERT` cells to verify.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import re
import json
import math
from typing import List, Tuple, Optional, Dict, Callable
from dataclasses import dataclass, field


---
## 1 · Verifiable Rewards — Math & Code

The cleanest RL signal comes from tasks where correctness is **objectively checkable**:
- **Math:** compare final numerical answer to ground truth
- **Code:** run tests, check output matches expected
- **Logic/formal:** automated theorem provers

### Why verifiable rewards are so powerful (DeepSeek-R1 insight)
- No reward model needed → no reward hacking via RM exploitation
- The reward is perfectly calibrated — no distribution shift
- Scales naturally: harder problems are still verifiable
- Enables long chain-of-thought: intermediate steps can be arbitrary as long as the answer is correct

### Reward design choices
| Design | Signal density | Risk |
|---|---|---|
| Binary (correct/wrong) | Very sparse | Hard to learn; no gradient on wrong answers |
| Partial credit (edit distance) | Dense | May reward near-misses that don't generalise |
| Format + correctness | Medium | Format hacking (correct format, wrong content) |
| Process reward (step-by-step) | Dense | Expensive to label; PRM training needed |


In [ ]:
def math_answer_reward(
    prediction: str,
    ground_truth: str,
    partial_credit: bool = False,
) -> float:
    """
    Extract the final number from prediction and ground_truth strings.
    Try patterns: '#### N', '= N', then last number in text.
    Return 1.0 for match within 1e-5, 0.0 otherwise.
    If partial_credit: return max(0, 1 - relative_error).
    """
    def extract_number(text: str) -> Optional[float]:
        # TODO: try patterns ['#### N', '= N', 'last number'], parse float
        raise NotImplementedError
    # TODO: extract both, compare
    raise NotImplementedError


def code_execution_reward(
    code_str: str,
    test_cases: List[Tuple[str, str]],
    timeout: float = 1.0,
) -> float:
    """
    exec code_str, call solution(input_str) for each test case.
    Return fraction of test cases passed. Handle exceptions gracefully (count as fail).
    """
    # TODO
    raise NotImplementedError


def format_reward(
    text: str,
    required_tags: List[str] = ("<think>", "</think>", "<answer>", "</answer>"),
) -> float:
    """
    Return 1.0 if all required_tags appear in text in order, 0.0 otherwise.
    Hint: maintain a position pointer, use str.find(tag, pos).
    """
    # TODO
    raise NotImplementedError


def composite_reward(
    prediction: str,
    ground_truth: str,
    test_cases: Optional[List[Tuple[str, str]]] = None,
    w_correctness: float = 1.0,
    w_format:      float = 0.2,
) -> Tuple[float, Dict[str, float]]:
    """r_total = w_correctness*r_correct + w_format*r_format"""
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
assert math_answer_reward("The answer is #### 42", "#### 42") == 1.0
assert math_answer_reward("#### 42", "#### 43") == 0.0
assert math_answer_reward("I think the answer is 3.14", "= 3.14") == 1.0
assert format_reward("<think>...</think><answer>42</answer>") == 1.0
assert format_reward("The answer is 42") == 0.0

code = """
def solution(x):
    return str(int(x) * 2)
"""
assert code_execution_reward(code, [("3","6"),("5","10")]) == 1.0
r, info = composite_reward("<think>step</think><answer>42</answer> #### 42", "#### 42")
assert "correctness" in info and "format" in info
print(f"math_answer_reward ✓  format_reward ✓  code_execution_reward ✓  composite_reward ✓")


---
## 2 · Outcome Reward Models (ORMs) vs Learned Reward Models

When verifiable rewards aren't available, we train a **reward model** (RM) on human preferences.

### Outcome Reward Model (ORM)
- Scores the **complete response** holistically
- Input: (prompt, full_response) → scalar
- Simple but sparse signal — no feedback on intermediate steps
- Used in InstructGPT, most production RLHF systems

### Reward model failure modes
1. **Out-of-distribution:** RM trained on distribution A, policy generates distribution B → unreliable scores
2. **Length bias:** RM often scores longer responses higher regardless of quality
3. **Style hacking:** model learns RM prefers certain words/phrases independent of content
4. **Overoptimisation:** Gao et al. (2023) showed RM score peaks then declines relative to ground truth

### Reward normalisation
Raw RM scores should be normalised before use as RL rewards:
- **Z-score:** `r_norm = (r - μ) / σ` — standardise across batch
- **Percentile:** `r_norm = rank(r) / n` — robust to outliers
- **Clipping:** `r_norm = clip(r, r_min, r_max)` — prevents extreme values from dominating


In [ ]:
class OutcomeRewardModel(nn.Module):
    """cat(prompt_emb, response_emb) → scalar reward. 2-hidden GELU MLP."""
    def __init__(self, embed_dim: int = 64, hidden: int = 128):
        super().__init__()
        # TODO: Sequential with GELU activations, output 1
        raise NotImplementedError
    def forward(self, prompt_emb, response_emb):
        # TODO: cat, net, squeeze(-1)
        raise NotImplementedError


def normalize_rewards_zscore(rewards: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """(r - mean) / std"""
    # TODO
    raise NotImplementedError


def normalize_rewards_percentile(rewards: torch.Tensor) -> torch.Tensor:
    """Rank / n. Hint: argsort twice gives ranks."""
    # TODO
    raise NotImplementedError


def clip_rewards(rewards: torch.Tensor,
                 percentile_low: float = 5.0,
                 percentile_high: float = 95.0) -> torch.Tensor:
    """Clip to [quantile(low/100), quantile(high/100)]."""
    # TODO
    raise NotImplementedError


def detect_length_bias(
    rewards: torch.Tensor,
    lengths: torch.Tensor,
    threshold: float = 0.3,
) -> Tuple[float, bool]:
    """
    Pearson correlation between rewards and lengths.
    Returns (corr, abs(corr) > threshold).
    """
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B, D = 16, 64
orm    = OutcomeRewardModel(D)
rewards = orm(torch.randn(B,D), torch.randn(B,D))
assert rewards.shape == (B,)

r_z = normalize_rewards_zscore(rewards.detach())
assert abs(r_z.mean().item()) < 1e-5 and abs(r_z.std().item() - 1.0) < 1e-5

r_pct = normalize_rewards_percentile(rewards.detach())
assert r_pct.min() > 0 and r_pct.max() <= 1.0

lengths = torch.randint(50, 500, (B,)).float()
corr, biased = detect_length_bias(lengths + torch.randn(B)*5, lengths)
assert biased, f"Expected bias, got corr={corr:.3f}"
assert not detect_length_bias(torch.randn(B), lengths)[1]
print("OutcomeRewardModel ✓  normalize zscore/percentile ✓  detect_length_bias ✓")


---
## 3 · Process Reward Models (PRMs)

**PRMs (Lightman et al., 2023 "Let's Verify Step by Step")** reward each **reasoning step** separately,  
not just the final answer.

### Motivation
- Math reasoning chains can be long (10–30 steps)
- ORM: only knows if the final answer is right — no gradient on intermediate steps
- PRM: tells the model *which step went wrong* — much denser signal

### PRM architecture
Same backbone as ORM, but applied at each **step boundary** (newline or `\n\n`):
```
prompt → step_1 → [PRM score_1] → step_2 → [PRM score_2] → ... → answer → [PRM score_T]
```

### Training PRMs (Monte Carlo estimation)
Without human labels on every step, use MC rollouts to estimate step quality:
1. Sample K completions from each step onwards
2. Step quality = fraction of completions that reach the correct answer
3. This is the **process reward** for that step

### Combining PRM with RL
The per-step PRM reward replaces (or augments) the terminal ORM reward in the RL loop.  
This gives a much denser training signal, especially for long chains.

### PRM vs ORM comparison
| | ORM | PRM |
|---|---|---|
| Reward density | Terminal only | Per step |
| Training data | Preference pairs | Step-level labels or MC |
| Signal quality | Coarse | Fine-grained |
| Used in | InstructGPT, most RLHF | OpenAI o1, DeepSeek-R1 |


In [ ]:
class ProcessRewardModel(nn.Module):
    """cat(prompt_emb, step_emb) → scalar score per step."""
    def __init__(self, embed_dim: int = 64, hidden: int = 128):
        super().__init__()
        # TODO: self.step_scorer — 2-layer GELU MLP, input=2*embed_dim, output=1
        raise NotImplementedError

    def score_steps(self, prompt_emb: torch.Tensor, step_embs: torch.Tensor) -> torch.Tensor:
        """
        prompt_emb: (D,) or (1, D)
        step_embs:  (T_steps, D)
        Returns: (T_steps,)
        Hint: expand prompt_emb to (T_steps, D) then cat with step_embs.
        """
        # TODO
        raise NotImplementedError


def mc_process_reward_estimation(completion_rewards: torch.Tensor) -> float:
    """Fraction of 1s in completion_rewards."""
    # TODO
    raise NotImplementedError


def aggregate_prm_rewards(step_scores: torch.Tensor, agg: str = "min") -> float:
    """agg ∈ {'min', 'mean', 'last', 'product'}"""
    # TODO
    raise NotImplementedError


def prm_weighted_reward(
    step_scores:  torch.Tensor,  # (T,)
    step_rewards: torch.Tensor,  # (T,)
    discount:     float = 0.99,
) -> torch.Tensor:
    """
    Potential-based reward shaping: r_aug_t = r_t + (prm_t - prm_{t-1})
    Use torch.diff with prepend=tensor([0.0]).
    """
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
D, T = 64, 5
prm    = ProcessRewardModel(D)
scores = prm.score_steps(torch.randn(D), torch.randn(T, D))
assert scores.shape == (T,)

mc = mc_process_reward_estimation(torch.tensor([1,1,1,0,1,1,0,1,1,0]))
assert abs(mc - 0.7) < 1e-5

s_good = torch.tensor([0.8,0.9,0.85,0.7,0.95])
s_bad  = torch.tensor([0.8,0.9,0.1,0.7,0.95])
assert aggregate_prm_rewards(s_bad, "min") < aggregate_prm_rewards(s_good, "min")

step_r = torch.zeros(T); step_r[-1] = 1.0
aug = prm_weighted_reward(s_good, step_r)
assert aug.shape == (T,)
print("ProcessRewardModel ✓  mc_process_reward ✓  aggregate_prm_rewards ✓  prm_weighted_reward ✓")


---
## 4 · Reward Shaping, Anti-Hacking Defences & Reward Ensembles

### Reward Shaping (Ng et al., 1999)
Any reward of the form $r'(s,a,s') = r(s,a,s') + \gamma \Phi(s') - \Phi(s)$  
**preserves the optimal policy** for any potential function $\Phi$.

This means we can add structure to the reward signal without changing the optimal behaviour:
- Use PRM scores as $\Phi$ → adds step-level guidance without biasing the optimum
- Use KL as $\Phi$ → equivalent to the KL-penalised objective in RLHF

### Reward Ensembles (Coste et al., 2023)
Train **multiple independent RMs** on different subsets of preference data.  
Use the **minimum** across the ensemble as the actual reward:
$$r_{ensemble}(x,y) = \min_i r_i(x,y)$$

This is a **pessimistic** reward — conservative under uncertainty.  
Prevents the policy from exploiting any single RM's blind spots.

### Length penalty
A common form of reward hacking is **length exploitation** (longer = higher RM score).  
Simple fix: subtract a length penalty from the reward:
$$r_{adj}(x,y) = r(x,y) - \alpha \cdot \max(0, |y| - L_{target})$$

### Reward clipping
Hard clip prevents extreme reward values from causing large gradient updates:
$$r_{clipped} = \text{clip}(r, r_{min}, r_{max})$$


In [ ]:
def ensemble_reward(
    rewards: torch.Tensor,  # (B, K)
    strategy: str = "min",
    temperature: float = 1.0,
) -> torch.Tensor:          # (B,)
    """
    'min': rewards.min(dim=1).values
    'mean': rewards.mean(dim=1)
    'softmin': softmax(-r/T) weighted sum — biases toward the minimum
    """
    # TODO
    raise NotImplementedError


def length_penalized_reward(
    rewards:      torch.Tensor,  # (B,)
    lengths:      torch.Tensor,  # (B,)
    target_len:   int   = 256,
    penalty_coef: float = 0.001,
) -> torch.Tensor:
    """r - penalty_coef * max(0, len - target_len). Use .clamp(min=0)."""
    # TODO
    raise NotImplementedError


def reward_with_confidence(
    rewards: torch.Tensor,     # (B, K)
    conf_threshold: float = 0.5,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Return (mean_reward (B,), high_conf_mask (B,)) where high_conf = std < threshold."""
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B, K = 16, 4
r_k  = torch.randn(B, K)
r_min  = ensemble_reward(r_k, "min")
r_mean = ensemble_reward(r_k, "mean")
assert r_min.shape == (B,) and (r_min <= r_mean).all()

lengths = torch.randint(100, 512, (B,))
r_raw   = torch.rand(B)
r_adj   = length_penalized_reward(r_raw, lengths, target_len=256)
assert (r_adj[lengths <= 256] == r_raw[lengths <= 256]).all()
assert (r_adj[lengths > 256]  <  r_raw[lengths > 256]).all()

_, conf = reward_with_confidence(r_k)
print("ensemble_reward ✓  length_penalized_reward ✓  reward_with_confidence ✓")


---
## 5 · Reward Overoptimisation — Gao et al. (2023)

**Key empirical finding (Gao et al., "Scaling Laws for Reward Model Overoptimization"):**

As the policy is optimised more aggressively against a proxy RM, the **proxy reward increases**  
but the **gold reward (true human preference) peaks and then decreases**.

The gap $r_{proxy} - r_{gold}$ grows as a function of KL from the reference model.

### The relationship (empirical)
$$r_{gold} \approx r_{proxy} - c \sqrt{D_{KL}(\pi_\theta \| \pi_{ref})}$$

where $c$ is a dataset-size-dependent constant. More RM training data → smaller $c$.

### Practical implications
1. **Early stopping:** don't optimise until KL convergence — stop when $D_{KL}$ reaches a threshold
2. **RM size:** larger, better-trained RMs have smaller $c$ — worth investing in RM quality
3. **Iterative RLHF:** refresh RM periodically with new on-policy labels
4. **Ensemble RMs:** reduces effective $c$ by averaging out individual RM biases


In [ ]:
def simulate_overoptimisation(
    n_steps:  int   = 200,
    beta:     float = 0.1,
    c:        float = 0.5,
    lr:       float = 0.05,
    seed:     int   = 42,
) -> Dict[str, List[float]]:
    """
    Scalar simulation of reward overoptimisation.
    Model:
      proxy_r = theta
      kl      = theta²
      gold_r  = proxy_r - c * sqrt(kl)
    Optimise: loss = -(proxy_r - beta * kl)

    Track history dict with keys: proxy, gold, kl, theta.
    """
    # TODO
    raise NotImplementedError


def find_optimal_kl_budget(c: float = 0.5, beta: float = 0.1) -> Dict[str, float]:
    """
    Optimal theta* for maximising (theta - beta*theta²):
      d/dθ [θ - β·θ²] = 1 - 2βθ = 0  →  θ* = 1/(2β)
    Return dict with theta_star, kl_star (= theta_star²), gold_at_kl_star.
    """
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
history = simulate_overoptimisation(n_steps=100, c=0.3, beta=0.05)
gold_arr = np.array(history["gold"])
assert len(gold_arr) == 100
peak_step = np.argmax(gold_arr)
assert gold_arr[peak_step] > gold_arr[-1], "Gold reward should peak and then decline"
print(f"simulate_overoptimisation ✓  gold peaks at step {peak_step}")

budget = find_optimal_kl_budget(c=0.3, beta=0.05)
assert "theta_star" in budget and "kl_star" in budget
print(f"find_optimal_kl_budget    ✓  {budget}")
